# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fayrouzhassan2000/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The action queue prioritizes content for human review based on the model's predicted likelihood of decline. Content with higher model scores is placed higher in the queue because it shows a stronger **decline-risk signal** in the evaluated data.

For each prioritized content item, we provide a short reason code based on observable data signals and map it to a suggested review action. These reason codes are intended to make the ranking easier for a human reviewer to interpret; they are **not causal explanations** of why performance declined.

The queue is therefore a **decision-support tool**, not an automated decision system. Higher-ranked items should be reviewed first, while the final content action remains with a human reviewer.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Data loading

import duckdb
import numpy as np
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks) AS gsc_clicks,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_sessions) AS ga4_sessions
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-03'
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

from datasets import load_dataset

content_ds = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    split="train"
)

content_df = content_ds.to_pandas()

content_df = content_df[
    [
        "client_hash_id",
        "content_hash_id",
        "content_type"
    ]
]

feature_df = features.merge(
    content_df,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

april = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS april_impressions
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE month = '2026-04'
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

march = feature_df.merge(
    april,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

march["is_declining_label"] = (
    march["april_impressions"] < march["gsc_impressions"]
)


march.drop(columns=["april_impressions"], axis = 1, inplace=True)

df = march.copy()
# =========================
# Handling Missing Values
# =========================

# 1. GA4 Sessions
# NaN means the client has no GA4 access
df["ga4_sessions"] = df["ga4_sessions"].fillna(0)


# 2. GSC
# NaN means there were no GSC impressions
# Create an indicator before imputation
df["gsc_avg_position_missing"] = (
    df["gsc_avg_position"].isna().astype(int)
)


feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "content_type",
    "gsc_avg_position_missing"
]

X = df[feature_cols]
y = df["is_declining_label"]

# Use client as the grouping variable
groups = df["client_hash_id"]

# Create grouped train/test split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

# Reuse the grouped train/test split
X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

numeric_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "gsc_avg_position_missing"
]

categorical_features = [
    "content_type"
]

# Numeric preprocessing
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Categorical preprocessing
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

# Full pipeline
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(random_state=42))
    ]
)

# Train
rf_pipeline.fit(X_train, y_train)

# Predict probabilities
K = 20

rf_scores = rf_pipeline.predict_proba(X_test)[:, 1]

# Rank by highest probability of decline
top_k_idx = np.argsort(rf_scores)[::-1][:K]

# Precision@20
precision_at_k_w6 = y_test.iloc[top_k_idx].mean()

print(f"Random Forest Precision@{K}: {precision_at_k_w6:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Random Forest Precision@20: 0.900


In [7]:
# =========================
# Build Ranked Action Queue
# =========================

# Get the test-set rows corresponding to the model scores
ranked_queue = df.iloc[test_idx].copy()

# Add model score
ranked_queue["decline_score"] = rf_scores

# Rank highest-risk content first
ranked_queue = ranked_queue.sort_values(
    "decline_score",
    ascending=False
).reset_index(drop=True)

ranked_queue["rank"] = ranked_queue.index + 1

# Keep the top-K items for the action queue
action_queue = ranked_queue.head(K).copy()

print(f"Action queue size: {len(action_queue)}")
action_queue.head(K)

Action queue size: 20


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,content_type,is_declining_label,gsc_avg_position_missing,decline_score,rank
0,client_157ffe4d4a595515,content_7014dc2e0f31c226,74.0,0.0,20.332738,0.0,comparison article,True,0,1.0,1
1,client_157ffe4d4a595515,content_ee560f2057ca3e7d,113.0,0.0,4.175096,0.0,comparison article,True,0,1.0,2
2,client_157ffe4d4a595515,content_e7340363e6ceefaa,18.0,0.0,5.825000,0.0,keyword article,False,0,1.0,3
3,client_157ffe4d4a595515,content_d243833882be0799,81.0,0.0,5.657407,0.0,comparison article,True,0,1.0,4
4,client_2b4306c3ed003f01,content_fcaeef7e7f978033,2.0,0.0,19.000000,0.0,keyword article,True,0,1.0,5
5,client_157ffe4d4a595515,content_5c50363b555f3f33,59.0,0.0,7.103333,0.0,comparison article,True,0,1.0,6
6,client_157ffe4d4a595515,content_084d132bb9bac884,74.0,0.0,7.386246,0.0,comparison article,False,0,1.0,7
7,client_c182d11e4862a37d,content_08f530afff096191,5.0,0.0,2.250000,0.0,feedly article,False,0,1.0,8
8,client_157ffe4d4a595515,content_e645ec261518ad35,250.0,0.0,5.889930,0.0,comparison article,True,0,1.0,9
9,client_157ffe4d4a595515,content_39eddc373f77ad83,127.0,0.0,5.709504,0.0,comparison article,True,0,1.0,10


In [12]:
# =========================
# Reason Codes + Suggested Actions
# =========================

def assign_reason_code(row):
    if row["gsc_avg_position_missing"] == 1:
        return "GSC_POSITION_UNAVAILABLE"
    elif row["gsc_impressions"] == 0:
        return "ZERO_GSC_IMPRESSIONS"
    elif row["gsc_clicks"] == 0:
        return "ZERO_GSC_CLICKS"
    else:
        return "MODEL_HIGH_DECLINE_RISK_SIGNAL"


def assign_action(reason_code):
    if reason_code == "GSC_POSITION_UNAVAILABLE":
        return "Review using available GSC signals"
    elif reason_code == "ZERO_GSC_IMPRESSIONS":
        return "Review discoverability and content performance"
    elif reason_code == "ZERO_GSC_CLICKS":
        return "Review content performance before deciding on refresh"
    else:
        return "Prioritize for human content review"


# Apply reason codes
action_queue["reason_code"] = action_queue.apply(
    assign_reason_code,
    axis=1
)

# Map each reason code to a suggested action
action_queue["suggested_action"] = action_queue["reason_code"].apply(
    assign_action
)

# All actions require human review
action_queue["human_review_required"] = True

# Flag tied model scores
action_queue["score_tie"] = (
    action_queue["decline_score"]
    .duplicated(keep=False)
)

# Display the final ranked action queue
action_queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "content_type",
        "decline_score",
        "score_tie",
        "reason_code",
        "suggested_action",
        "human_review_required"
    ]
]

,rank,client_hash_id,content_hash_id,content_type,decline_score,score_tie,reason_code,suggested_action,human_review_required
0,1,client_157ffe4d4a595515,content_7014dc2e0f31c226,comparison article,1.0,True,ZERO_GSC_CLICKS,Review content performance before deciding on ...,True
1,2,client_157ffe4d4a595515,content_ee560f2057ca3e7d,comparison article,1.0,True,ZERO_GSC_CLICKS,Review content performance before deciding on ...,True
2,3,client_157ffe4d4a595515,content_e7340363e6ceefaa,keyword article,1.0,True,ZERO_GSC_CLICKS,Review content performance before deciding on ...,True
3,4,client_157ffe4d4a595515,content_d243833882be0799,comparison article,1.0,True,ZERO_GSC_CLICKS,Review content performance before deciding on ...,True
4,5,client_2b4306c3ed003f01,content_fcaeef7e7f978033,keyword article,1.0,True,ZERO_GSC_CLICKS,Review content performance before deciding on ...,True
5,6,client_157ffe4d4a595515,content_5c50363b555f3f33,comparison article,1.0,True,ZERO_GSC_CLICKS,Review content performance before deciding on ...,True
6,7,client_157ffe4d4a595515,content_084d132bb9bac884,comparison article,1.0,True,ZERO_GSC_CLICKS,Review content performance before deciding on ...,True
7,8,client_c182d11e4862a37d,content_08f530afff096191,feedly article,1.0,True,ZERO_GSC_CLICKS,Review content performance before deciding on ...,True
8,9,client_157ffe4d4a595515,content_e645ec261518ad35,comparison article,1.0,True,ZERO_GSC_CLICKS,Review content performance before deciding on ...,True
9,10,client_157ffe4d4a595515,content_39eddc373f77ad83,comparison article,1.0,True,ZERO_GSC_CLICKS,Review content performance before deciding on ...,True


### Archetype → Action mapping

The action mapping translates observable performance patterns into review-oriented actions. These mappings are directional recommendations rather than causal conclusions.

| Archetype / observed signal    | Suggested action                                                                       | Human review |
| ------------------------------ | -------------------------------------------------------------------------------------- | ------------ |
| High model decline-risk signal | Prioritize the content for review                                                      | Required     |
| Zero GSC clicks                | Review content performance and search visibility before deciding on a refresh          | Required     |
| Zero GSC impressions           | Review discoverability and content relevance                                           | Required     |
| GSC position unavailable       | Review using the available performance signals                                         | Required     |
| Tied model scores              | Treat items as the same priority tier rather than assuming meaningful rank differences | Required     |

The model does not automatically select, rewrite, delete, or publish content. The suggested actions are intended to help a human reviewer decide what to investigate first.


In [13]:
# =========================
# Archetype → Action Mapping
# =========================

import pandas as pd

archetype_action_map = {
    "MODEL_HIGH_DECLINE_RISK_SIGNAL": {
        "archetype": "High decline-risk signal",
        "suggested_action": "Prioritize for human content review"
    },
    "ZERO_GSC_CLICKS": {
        "archetype": "Zero GSC clicks",
        "suggested_action": "Review content performance before deciding on refresh"
    },
    "ZERO_GSC_IMPRESSIONS": {
        "archetype": "Zero GSC impressions",
        "suggested_action": "Review discoverability and content performance"
    },
    "GSC_POSITION_UNAVAILABLE": {
        "archetype": "GSC position unavailable",
        "suggested_action": "Review using available GSC signals"
    }
}

archetype_mapping = pd.DataFrame(
    [
        {
            "reason_code": reason_code,
            "archetype": values["archetype"],
            "suggested_action": values["suggested_action"],
            "human_review_required": True
        }
        for reason_code, values in archetype_action_map.items()
    ]
)

archetype_mapping

,reason_code,archetype,suggested_action,human_review_required
0,MODEL_HIGH_DECLINE_RISK_SIGNAL,High decline-risk signal,Prioritize for human content review,True
1,ZERO_GSC_CLICKS,Zero GSC clicks,Review content performance before deciding on ...,True
2,ZERO_GSC_IMPRESSIONS,Zero GSC impressions,Review discoverability and content performance,True
3,GSC_POSITION_UNAVAILABLE,GSC position unavailable,Review using available GSC signals,True


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

The FlyRank output is intended to help content teams **prioritize content for human review** based on model-based decline-risk signals.

It can be used to:

* Identify which content items should be reviewed first.
* Highlight observable performance signals for further investigation.
* Support decisions about whether a content item may need a refresh or additional analysis.
* Help prioritize limited editorial resources.

The model is a **decision-support tool**, not an autonomous decision-maker. Final content decisions remain with a human reviewer.

### Limits

The model has several important limitations:

* A high decline score represents a stronger model-based risk signal; it does not guarantee that the content will decline.
* Reason codes describe observable data signals and are not causal explanations of performance changes.
* The model does not guarantee that a suggested action will improve traffic, clicks, impressions, or other business outcomes.
* Missing or unavailable data can affect the reliability of some signals. For example, zero GA4 sessions may indicate unavailable GA4 access rather than zero actual traffic.
* Multiple items may have the same model score, so small rank differences within a tied group should not be interpreted as meaningful differences in risk.
* Performance may differ when the model is applied to new data or a different evaluation setting.

The output should therefore be used as a **prioritization aid for human review**, rather than as a production decision engine.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.




## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review rules

Every item in the ranked queue requires human review before any content action is taken.

The reviewer should:

1. Verify that the underlying GSC and GA4 signals are available and interpreted correctly.
2. Check the recent performance of the content and compare it with relevant historical context.
3. Review the content itself and consider factors that are not represented in the model features.
4. Treat reason codes as observable signals for investigation, not as explanations of causality.
5. Consider the potential effort and value of the proposed action before making a decision.
6. Record the final decision and the rationale for taking or rejecting the suggested action.

### No-go list

The model should **not** autonomously:

* Delete or unpublish content.
* Rewrite or publish content.
* Change URLs, redirects, or canonical settings.
* Remove content solely because it has a high decline score.
* Make major SEO or editorial changes without human approval.
* Treat a high model score as proof that a specific intervention will improve performance.
* Make irreversible business or editorial decisions based only on the model output.

The model's role stops at **prioritization and decision support**. A human remains responsible for interpreting the evidence and approving any action.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring

The model should be monitored periodically using the same evaluation setup used for validation.

Key monitoring signals include:

* **Precision@20:** Track whether the model continues to prioritize declining content effectively.
* **Score distribution:** Check whether the distribution of model scores changes substantially over time.
* **Input data quality:** Monitor missing values and the availability of GSC and GA4 signals.
* **Content mix:** Check whether the distribution of content types changes substantially compared with the evaluation data.
* **Score ties:** Track whether a large proportion of the ranked queue receives identical scores, since this can reduce meaningful differentiation within the queue.

### Retrain / review triggers

A model review or retraining should be considered when:

* Precision@20 shows a sustained deterioration across multiple evaluation periods.
* The input data distribution changes substantially from the data used for validation.
* Important features have materially different missingness or availability patterns.
* The content mix changes enough that the original validation data may no longer represent the current workload.
* The model produces increasingly tied or poorly differentiated rankings.
* A meaningful change in the data pipeline or feature definitions occurs.

A single unusual observation should not automatically trigger retraining. These signals should prompt investigation first, followed by retraining only when the evidence indicates that the model or its validation setup needs updating.


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# =========================
# Lightweight Monitoring Checks
# =========================

monitoring_summary = {
    "precision_at_20": precision_at_k_w6,
    "queue_size": len(action_queue),
    "unique_decline_scores": action_queue["decline_score"].nunique(),
    "tied_rows": int(action_queue["score_tie"].sum()),
    "tie_rate": action_queue["score_tie"].mean(),
    "missing_gsc_position_rate": (
        action_queue["gsc_avg_position_missing"].mean()
    ),
    "zero_gsc_click_rate": (
        (action_queue["gsc_clicks"] == 0).mean()
    ),
    "zero_gsc_impression_rate": (
        (action_queue["gsc_impressions"] == 0).mean()
    )
}

monitoring_summary

{'precision_at_20': np.float64(0.9),
 'queue_size': 20,
 'unique_decline_scores': 1,
 'tied_rows': 20,
 'tie_rate': np.float64(1.0),
 'missing_gsc_position_rate': np.float64(0.0),
 'zero_gsc_click_rate': np.float64(1.0),
 'zero_gsc_impression_rate': np.float64(0.0)}

In [15]:
# =========================
# Score Tie Check
# =========================

if action_queue["score_tie"].mean() > 0.5:
    print(
        "Monitoring flag: More than 50% of the queue has tied model scores."
    )
else:
    print(
        "No major score-tie flag in the current queue."
    )

Monitoring flag: More than 50% of the queue has tied model scores.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exported artifacts

The ranked action queue is exported as a CSV file so that it can be regenerated from the executed notebook and reused in the research paper.

The exported queue contains:

* Rank and model-based decline score.
* Client and content identifiers.
* Content type.
* Observable reason codes.
* Suggested actions.
* Human-review requirements.
* Score-tie information.

The queue is intentionally kept out of version control because it contains data-derived records. The notebook remains the reproducible source for regenerating the queue.

No production deployment or automated content changes are performed by this notebook.


In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# =========================
# Export Ranked Action Queue
# =========================

import os

output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

queue_path = os.path.join(
    output_dir,
    "w07_ranked_action_queue.csv"
)

action_queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "content_type",
        "decline_score",
        "score_tie",
        "reason_code",
        "suggested_action",
        "human_review_required"
    ]
].to_csv(queue_path, index=False)

print(f"Ranked action queue exported to: {queue_path}")


Ranked action queue exported to: work/outputs/w07_ranked_action_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.